# ODI to Databricks Migration

## Converted from TARGET ODI SQL file
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

This notebook contains the converted SQL logic from the original ODI session.
It performs an INSERT operation from `HR.DEPARTMENTS` into `HR.TRG_DEP`.

In [ ]:
import dbutils

dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

The following parameters control the execution of this ETL process. These are defined as Databricks widgets.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  '${DATASOURCE_NUM_ID}' AS datasource_num_id,
  '${ETL_PROC_WID}' AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Staging Table (C$)

This section would typically handle the creation and population of C$ staging tables.
The provided ODI source does not define explicit C$ tables for this task.

## Flow Table (I$)

This section would typically handle the creation and population of I$ flow tables.
The provided ODI source does not define explicit I$ tables for this task.

## Error / Check Tables (E$, SNP_CHECK_TAB)

This section handles the creation and management of error logging and check tables.
The provided ODI source does not define explicit E$ tables or use `SNP_CHECK_TAB` for this task.

## PK Violation Detection

This section would typically contain logic to identify and handle Primary Key violations.
The provided ODI source does not include such logic for this task.

## Mark Records for Update

This section would typically mark records in a flow table for update based on existing records in the target table.
The provided ODI source performs a direct INSERT, so this step is not applicable.

## DML into Target: `workspace.hr.trg_dep`

### SCEN_TASK_NO in {10}: Initial Setup (No explicit SQL in source)
### SCEN_TASK_NO in {20}: Pre-processing (No explicit SQL in source)
### SCEN_TASK_NO in {30}: Data Preparation (No explicit SQL in source)
### Original `INSERT` statement (Converted to Spark SQL)

In [ ]:
%sql
INSERT
  INTO workspace.hr.trg_dep
  (
    department_id,
    department_name,
    manager_id,
    location_id
  )
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Record Counts after INSERT

In [ ]:
%sql
SELECT
  COUNT(*) AS total_records_in_trg_dep
FROM
  workspace.hr.trg_dep;

## Optimize Target

This section optimizes the target Delta table for query performance.

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_dep ZORDER BY (department_id);

## Cleanup

This section handles the cleanup of temporary staging and flow tables.
No explicit C$ or I$ tables were created for this task, so no cleanup is required.

## Validation

Final checks to validate the data loaded into the target table.

In [ ]:
%sql
SELECT
  department_id,
  department_name,
  manager_id,
  location_id
FROM
  workspace.hr.trg_dep
ORDER BY
  department_id
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All Oracle schema (`HR`) and table names (`TRG_DEP`, `DEPARTMENTS`) have been converted to lowercase and prefixed with `workspace.` (e.g., `workspace.hr.trg_dep`, `workspace.hr.departments`).
2.  **`/*+ APPEND PARALLEL */` Hint:** The Oracle hint has been removed as it is not applicable in Databricks Spark SQL for Delta tables.
3.  **Column Casing:** Column names in the `INSERT` statement have been converted to lowercase to align with general Spark SQL best practices and case-insensitivity unless quoted.
4.  **Widget Parameters:** Standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been added as Databricks widgets, although they are not directly used in the provided SQL. Default values are set.
5.  **Empty SCEN_TASK_NOs:** The empty `SCEN_TASK_NO {10}`, `{20}`, `{30}` from the original source are noted as comments in the relevant SQL cell.
6.  **`OPTIMIZE ZORDER BY`:** An `OPTIMIZE` statement with `ZORDER BY (department_id)` has been added for potential performance benefits, preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` as required.
7.  **Missing DDL:** The original ODI source did not include `CREATE TABLE` DDL for `TRG_DEP` or `DEPARTMENTS`. It is assumed that these tables already exist or will be created externally in Databricks with appropriate Spark SQL data types (e.g., `BIGINT`, `STRING`, `TIMESTAMP`). If `TRG_DEP` does not exist, this INSERT statement will fail.

---